In [1]:
from pathlib import Path
import pandas as pd
import sys
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import KFold
import seaborn as sns
from scipy.stats import norm
from sklearn.metrics import roc_auc_score

# путь до корня проекта
PROJECT_ROOT = Path("../../").resolve()

# добавляем в PYTHONPATH
sys.path.append(str(PROJECT_ROOT))

from src.assembler.io import load_feature_names_from_txt

In [2]:
DATA_PATH = Path("../../data/raw/clean_dataset.csv")
DATA_PATH2 = Path("../../data/raw/dataset_2025-03-01_2026-03-29_external.csv")
FEATURES_PATH = Path("../../data/feature_groups/features_group_5.txt")
OUTPUT_DIR = Path("../../notebook_outputs/group_5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

*************

In [3]:
block_df = pd.read_csv(DATA_PATH, low_memory=False)
feature_names = load_feature_names_from_txt(FEATURES_PATH)
df = block_df[feature_names].copy()

print("Block shape:", block_df.shape)
pd.set_option('display.max_columns', None)


Block shape: (18887, 93)


In [4]:
df["row_id"]

0            0
1            1
2            2
3            3
4            4
         ...  
18882    18882
18883    18883
18884    18884
18885    18885
18886    18886
Name: row_id, Length: 18887, dtype: int64

In [5]:
df[~df["lead_Ширина"].isna()]["lead_Ширина"].describe()

count    12103.000000
mean        29.706106
std          7.957257
min          2.000000
25%         30.000000
50%         30.000000
75%         31.000000
max        400.000000
Name: lead_Ширина, dtype: float64

In [6]:
def width_category(x):
    if pd.isna(x):
        return "unknown"

    if x < 27:
        return "small"
    elif x <= 33:
        return "medium"
    elif x <= 40:
        return "large"
    else:
        return "very_large"

df["width_cat"] = df["lead_Ширина"].apply(width_category)
mapping = {
    "very_large": "high",
    "small": "high",
    "medium": "mid",
    "large": "mid",
    "unknown": "unknown"
}

df["width_cat"] = df["width_cat"].map(mapping)
df["width_is_missing"] = df["lead_Ширина"].isna().astype(int)
df = df.drop(columns=["lead_Ширина"])


In [7]:
df["width_cat"].isna().sum(), df["width_is_missing"].isna().sum()

(np.int64(0), np.int64(0))

In [8]:
df.groupby("width_cat")["buyout_flag"].agg(["count", "mean"])

,count,mean
width_cat,,
high,2402,0.893006
mid,9235,0.837466
unknown,6329,0.790488


***

In [9]:
df[~df["lead_Линейная высота (см)"].isna()]["lead_Линейная высота (см)"].describe()

count    3143.000000
mean       11.618836
std        12.790076
min         2.000000
25%         5.000000
50%         8.000000
75%        13.000000
max       430.000000
Name: lead_Линейная высота (см), dtype: float64

In [10]:
# обрезаем хвост (например, до 99 перцентиля)
"""
upper = df["lead_Линейная высота (см)"].quantile(0.99)
df["lead_height_clipped"] = df["lead_Линейная высота (см)"].clip(upper=upper)
#df["lead_height_log"] = np.log1p(df["lead_height_clipped"])
def height_bin(x):
    if pd.isna(x):
        return "unknown"
    elif x <= 5:
        return "small"
    elif x <= 13:
        return "medium"
    elif x <= 30:
        return "large"
    else:
        return "very_large"

df["lead_height_bin"] = df["lead_height_clipped"].apply(height_bin)"""


'\nupper = df["lead_Линейная высота (см)"].quantile(0.99)\ndf["lead_height_clipped"] = df["lead_Линейная высота (см)"].clip(upper=upper)\n#df["lead_height_log"] = np.log1p(df["lead_height_clipped"])\ndef height_bin(x):\n    if pd.isna(x):\n        return "unknown"\n    elif x <= 5:\n        return "small"\n    elif x <= 13:\n        return "medium"\n    elif x <= 30:\n        return "large"\n    else:\n        return "very_large"\n\ndf["lead_height_bin"] = df["lead_height_clipped"].apply(height_bin)'

In [11]:
#df.groupby("lead_height_bin")["buyout_flag"].agg(["count", "mean"])


In [12]:
"""df["lead_height_known"] = df["lead_Линейная высота (см)"].notna().astype(int)
df["lead_height_clipped"] = df["lead_Линейная высота (см)"].clip(upper=upper)
df["lead_height_log"] = np.log1p(df["lead_height_clipped"])
df["lead_height_log"] = df["lead_height_log"].fillna(-1)"""
# Есть более жирный признак
df = df.drop(columns=["lead_Линейная высота (см)"])
                      #"lead_height_clipped"])

****

In [13]:
df[~df["lead_Вид оплаты"].isna()]["lead_Вид оплаты"].value_counts()

lead_Вид оплаты
Наложенный платеж         18080
Оплата онлайн               601
Оплата на карту             180
Оплата Золотой Короной        4
Name: count, dtype: int64

In [14]:
def payment_type(x):
    if pd.isna(x):
        return "unknown"
    elif x == "Наложенный платеж":
        return "cash_on_delivery"
    else:
        return "online"

df["lead_payment_type"] = df["lead_Вид оплаты"].apply(payment_type)
df = df.drop(columns=["lead_Вид оплаты"])

In [15]:
df.groupby("lead_payment_type")["buyout_flag"].agg(["count", "mean"])

,count,mean
lead_payment_type,,
cash_on_delivery,17175,0.822416
online,769,0.975293
unknown,22,0.318182


*****

In [16]:
# returned_dt -утечка, просто удаляем
df = df.drop(columns=["returned_ts"])

************

In [17]:
df[~df["lead_Служба доставки"].isna()]["lead_Служба доставки"].value_counts()


lead_Служба доставки
СДЭК до ПВЗ      13118
Почта             3217
СДЭК до Двери     2510
Самовывоз           25
Курьер ЕМС           2
Name: count, dtype: int64

In [18]:
def delivery_type(x):
    if pd.isna(x):
        return "unknown"
    elif x == "СДЭК до ПВЗ":
        return "pickup_point"
    elif x in ["СДЭК до Двери", "Курьер ЕМС"]:
        return "door_delivery"
    elif x == "Почта":
        return "post"
    elif x == "Самовывоз":
        return "pickup_point"  # логично объединить
    else:
        return "other"  # на случай мусора

df["lead_delivery_type"] = df["lead_Служба доставки"].apply(delivery_type)
#df = df.drop(columns=["lead_Служба доставки"])

In [19]:
df.groupby("lead_delivery_type")["buyout_flag"].agg(["count", "mean"])

,count,mean
lead_delivery_type,,
door_delivery,2418,0.811001
pickup_point,12611,0.847197
post,2922,0.764887
unknown,15,0.133333


*********

In [20]:
df[~df["lead_Компания Отправитель"].isna()]["lead_Компания Отправитель"].value_counts()

lead_Компания Отправитель
ООО ТехПродЗдрав    18581
ООО Здоров            285
Name: count, dtype: int64

In [21]:
df.groupby("lead_Компания Отправитель")["buyout_flag"].agg(["count", "mean"])

,count,mean
lead_Компания Отправитель,,
ООО Здоров,279,0.878136
ООО ТехПродЗдрав,17667,0.828154


In [22]:
# разница мала, и большой дисбаланс значений
df = df.drop(columns=["lead_Компания Отправитель"])

*********

In [23]:
#lead_group_id
df[~df["lead_group_id"].isna()]["lead_group_id"].value_counts()

lead_group_id
700242    6107
546538    6003
0         4257
700246    1817
708650     703
Name: count, dtype: int64

In [24]:
df.groupby("lead_group_id")["buyout_flag"].agg(["count", "mean"])

,count,mean
lead_group_id,,
0,4238,0.848749
546538,5586,0.832975
700242,5683,0.804857
700246,1758,0.818544
708650,701,0.883024


In [25]:

def target_encode_oof(df, col, target, n_splits=5):
    global_mean = df[target].mean()
    result = pd.Series(index=df.index, dtype=float)
    #todo исправить утечку
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for train_idx, val_idx in kf.split(df):
        train = df.iloc[train_idx]
        val = df.iloc[val_idx]

        means = train.groupby(col)[target].mean().astype(float)

        mapped = val[col].map(means)
        mapped = pd.to_numeric(mapped, errors="coerce")

        result.iloc[val_idx] = mapped.to_numpy()

    return result.fillna(global_mean)

In [26]:
#низкая кардинальность, всего ~5 значений, достаточное количество наблюдений
# есть порядок 0.80 → 0.83 → 0.85 → 0.88
df["lead_group_id_missing"] = df["lead_group_id"].isna().astype(int)
df["lead_group_id"] = df["lead_group_id"].fillna("unknown")
df["lead_group_quality"] = target_encode_oof(df, "lead_group_id", "buyout_flag")

#df = df.drop(columns=["lead_group_id"])

*********

In [27]:

df[~df["lead_Масса (гр)"].isna()]["lead_Масса (гр)"].describe()

count     3150.000000
mean       896.606349
std        965.804069
min          1.000000
25%        384.000000
50%        664.000000
75%       1150.000000
max      16950.000000
Name: lead_Масса (гр), dtype: float64

In [28]:
# слишком много пропусков
df["lead_mass_known"] = df["lead_Масса (гр)"].notna().astype(int)

# Ассиметрия, большие выбросы и разбросы
# есть ли масса бинарный сигнал, насколько большая  количественный сигнал
df["lead_mass_log"] = np.log1p(df["lead_Масса (гр)"])

df["lead_mass_log"] = df["lead_mass_log"].fillna(0)
df = df.drop(columns=["lead_Масса (гр)"])

***********

In [29]:
"""df["lead_closed_dt"] = pd.to_datetime(df["lead_closed_at"], unit="s",
                                      utc=True)

len(df[~df["lead_closed_dt"].isna()]["lead_closed_dt"])
df.groupby(df["lead_closed_dt"].notna())["buyout_flag"].mean()"""

'df["lead_closed_dt"] = pd.to_datetime(df["lead_closed_at"], unit="s",\n                                      utc=True)\n\nlen(df[~df["lead_closed_dt"].isna()]["lead_closed_dt"])\ndf.groupby(df["lead_closed_dt"].notna())["buyout_flag"].mean()'

In [30]:
# buyout → приводит к закрытию лида, утечка, 0.86!
df = df.drop(columns=["lead_closed_at"])

**********

In [31]:
df["contact_created_dt"] = pd.to_datetime(df["contact_created_at"],
                                          unit="s",
                                      utc=True)
df["contact_updated_dt"] = pd.to_datetime(df["contact_updated_at"],
                                          unit="s",
                                      utc=True)

In [32]:
df["lead_created_dt"] = pd.to_datetime(df["lead_created_at"], unit="s", utc=True)
(df["contact_updated_dt"] > df["lead_created_dt"]).mean()

np.float64(0.9994175888177053)

In [33]:
(df["contact_created_dt"] > df["lead_created_dt"]).mean()

np.float64(0.26293217557049825)

In [34]:
df["contact_to_lead_hours"] = (
    df["lead_created_dt"] - df["contact_created_dt"]
).dt.total_seconds() / 3600

df["contact_to_lead_hours"] = df["contact_to_lead_hours"].clip(lower=0)

df["contact_missing"] = df["contact_created_dt"].isna().astype(int)

df["contact_to_lead_hours"] = df["contact_to_lead_hours"].fillna(0)

df["contact_hour"] = df["contact_created_dt"].dt.hour
df["contact_dayofweek"] = df["contact_created_dt"].dt.dayofweek
df["contact_month"] = df["contact_created_dt"].dt.month
df["contact_is_weekend"] = (df["contact_created_dt"].dt.dayofweek >= 5).astype(int)

df["contact_month"] = df["contact_month"].fillna(-1)
df["contact_hour"] = df["contact_hour"].fillna(-1)
df["contact_dayofweek"] = df["contact_dayofweek"].fillna(-1)
df["contact_is_weekend"] = df["contact_is_weekend"].fillna(-1)

# contact_updated_dt признак формируется из события будущего на момент
# предсказания его не существует - удалить
df = df.drop(columns=[
    "contact_updated_dt",
    "contact_updated_at",
    "contact_created_dt",
    "contact_created_at"
])

****

In [35]:
df[~df["lead_utm_medium"].isna()]["lead_utm_medium"].value_counts()


lead_utm_medium
cpc                             11975
Неизвестно                        547
Bloger                             26
cpc__rt_view-yes_lead-no_all       15
article_direct                      4
cpm                                 2
organic                             1
Name: count, dtype: int64

In [36]:
# Смотрим платный трафик или бесплатный CPC = Cost Per Click и CPM = Cost Per Mille (1000 показов). Это индикатор платного привлечения клиента, тут еще можно поэкспериментировать
# если время будет
df["is_paid_traffic"] = df["lead_utm_medium"].astype(str).str.lower().str.contains("cpc|cpm").astype(int)
df = df.drop(columns=["lead_utm_medium"])

In [37]:
df.groupby("is_paid_traffic")["buyout_flag"].mean()

is_paid_traffic
0    0.879922
1    0.798002
Name: buyout_flag, dtype: object

********

In [38]:
df[~df["lead_Категория и варианты выбора"].isna()]["lead_Категория и варианты выбора"].value_counts()


lead_Категория и варианты выбора
S                5602
I                2324
D                1367
C                 356
Нет категории     190
Name: count, dtype: int64

In [39]:
df.groupby("lead_Категория и варианты выбора")["buyout_flag"].mean()

lead_Категория и варианты выбора
C                0.794393
D                0.823973
I                0.778654
S                0.817967
Нет категории    0.790323
Name: buyout_flag, dtype: object

In [40]:
# отражает “массовость категории”
df["lead_Категория и варианты выбора"] = df["lead_Категория и варианты выбора"].fillna("unknown").replace({"Нет категории": "unknown"})
# считаю по всему dataset, утечка есть, но слабая и обычно не влияет на результат
# существенно. В идеале считать на train, но думаю избыточно
freq = df["lead_Категория и варианты выбора"].value_counts(normalize=True)
df["lead_category_freq"] = df["lead_Категория и варианты выбора"].map(freq)

In [41]:
df = df.drop(columns=["lead_Категория и варианты выбора"])

In [42]:
df["lead_category_freq"].value_counts()

lead_category_freq
0.489120    9238
0.296606    5602
0.123048    2324
0.072378    1367
0.018849     356
Name: count, dtype: int64

************

In [43]:
# received_dt -утечка, просто удаляем, факт завершённой сделки
df = df.drop(columns=["received_ts"])

************

In [44]:
df[~df["lead_Модель телефона"].isna()]["lead_Модель телефона"].value_counts()


lead_Модель телефона
Смартфон             7924
Не удалось узнать    5739
Кнопочный             272
Name: count, dtype: int64

In [45]:
df.groupby("lead_Модель телефона")["buyout_flag"].mean()
#df["lead_Модель телефона"].isna().mean()

lead_Модель телефона
Кнопочный             0.76834
Не удалось узнать     0.81006
Смартфон             0.821717
Name: buyout_flag, dtype: object

In [46]:
df["lead_Модель телефона"] = df["lead_Модель телефона"].fillna("unknown")
df["is_feature_phone"] = (df["lead_Модель телефона"] == "Кнопочный").astype(int)
# todo сделать target encoding


In [47]:
df = df.drop(columns=["lead_Модель телефона"])

In [48]:
df.groupby("is_feature_phone")["buyout_flag"].mean()

is_feature_phone
0    0.82922
1    0.76834
Name: buyout_flag, dtype: object

*********

In [49]:
"""col1 = "lead_Дата перехода Передан в доставку"
col2 = "handed_to_delivery_dt"
df[col1] = pd.to_datetime(df[col1], unit="s", utc=True)
df[col2] = pd.to_datetime(df["handed_to_delivery_ts"], unit="s", utc=True)
df[~df[col1].isna() & ~df[col2].isna()][[col1, col2]]"""

'col1 = "lead_Дата перехода Передан в доставку"\ncol2 = "handed_to_delivery_dt"\ndf[col1] = pd.to_datetime(df[col1], unit="s", utc=True)\ndf[col2] = pd.to_datetime(df["handed_to_delivery_ts"], unit="s", utc=True)\ndf[~df[col1].isna() & ~df[col2].isna()][[col1, col2]]'

In [50]:

#основной признак, fallback на handed_to_delivery_ts не нужен, плохо заполненый
# lead_Дата перехода Передан в доставку" просто удаляем
# Итого признак "lead_Дата перехода Передан в доставку" разреженный
# дубль по смыслу handed_to_delivery_dt, удаляем
df = df.drop(columns=["lead_Дата перехода Передан в доставку"])


**********

In [51]:
# handed_to_delivery_ts
# handed_to_delivery_dt
#Нельзя использовать. передача в доставку происходит позже оформления
#в день оформления ты ещё не знаешь, когда заказ передадут в доставку
#значит это future information

"""
# --- 1. даты ---
df["lead_created_dt"] = pd.to_datetime(df["lead_created_at"], unit="s", utc=True, errors="coerce")
df["handed_to_delivery_dt"] = pd.to_datetime(df["handed_to_delivery_ts"], unit="s", utc=True, errors="coerce")

# --- 2. скорость передачи в доставку ---
df["lead_to_delivery_days"] = (
    df["handed_to_delivery_dt"] - df["lead_created_dt"]
).dt.total_seconds() / 86400

# защита от отрицательных значений
df["lead_to_delivery_days"] = df["lead_to_delivery_days"].clip(lower=0)

# --- 3. LOO-прокси по службе доставки  ---
group_sum = df.groupby("lead_Служба доставки")["lead_to_delivery_days"].transform("sum")
group_count = df.groupby("lead_Служба доставки")["lead_to_delivery_days"].transform("count")

df["delivery_speed_service_mean_loo"] = (
    group_sum - df["lead_to_delivery_days"]
) / (group_count - 1)

global_mean = df["lead_to_delivery_days"].mean()
df["delivery_speed_service_mean_loo"] = df["delivery_speed_service_mean_loo"].fillna(global_mean)

# --- 4. лог-признак ---
df["lead_to_delivery_days_log"] = np.log1p(df["lead_to_delivery_days"])

# --- 5. missing flag ---
df["lead_to_delivery_days_missing"] = df["lead_to_delivery_days"].isna().astype(int)

# --- 6. заполнение ---
df["lead_to_delivery_days_log"] = df["lead_to_delivery_days_log"].fillna(-1)

# --- 7. удаление лишнего (ВАЖНО: убираем leakage) ---
df = df.drop(columns=[
    "lead_created_dt",
    "handed_to_delivery_dt",
    "lead_to_delivery_days",
    "lead_Служба доставки"
    #"handed_to_delivery_ts"
])"""

'\n# --- 1. даты ---\ndf["lead_created_dt"] = pd.to_datetime(df["lead_created_at"], unit="s", utc=True, errors="coerce")\ndf["handed_to_delivery_dt"] = pd.to_datetime(df["handed_to_delivery_ts"], unit="s", utc=True, errors="coerce")\n\n# --- 2. скорость передачи в доставку ---\ndf["lead_to_delivery_days"] = (\n    df["handed_to_delivery_dt"] - df["lead_created_dt"]\n).dt.total_seconds() / 86400\n\n# защита от отрицательных значений\ndf["lead_to_delivery_days"] = df["lead_to_delivery_days"].clip(lower=0)\n\n# --- 3. LOO-прокси по службе доставки  ---\ngroup_sum = df.groupby("lead_Служба доставки")["lead_to_delivery_days"].transform("sum")\ngroup_count = df.groupby("lead_Служба доставки")["lead_to_delivery_days"].transform("count")\n\ndf["delivery_speed_service_mean_loo"] = (\n    group_sum - df["lead_to_delivery_days"]\n) / (group_count - 1)\n\nglobal_mean = df["lead_to_delivery_days"].mean()\ndf["delivery_speed_service_mean_loo"] = df["delivery_speed_service_mean_loo"].fillna(global_me

In [52]:
df = df.drop(columns=["handed_to_delivery_ts"])

******

In [53]:
# ID рекламной кампании
df[~df["lead_utm_campaign"].isna()]["lead_utm_campaign"].value_counts()

lead_utm_campaign
116543546     685
115509254     560
Неизвестно    547
118448375     376
700615408     339
             ... 
707281666       1
707338875       1
707363652       1
707177354       1
707306601       1
Name: count, Length: 1223, dtype: int64

In [54]:
# Уберем мусор
df["lead_utm_campaign"] = (df["lead_utm_campaign"].replace
                           (["{campaing_id}", "Неизвестно"], np.nan))

# Признак сильно разреженный, отмечаем где не заполнен
df["lead_utm_campaign_missing"] = df["lead_utm_campaign"].isna().astype(int)

# В теории можно Target Encoding, но пока простейший вариант, хвост
# сократить
top_k = 30  # можно 20–50
top_values = df["lead_utm_campaign"].value_counts().head(top_k).index

df["lead_utm_campaign_grouped"] = df["lead_utm_campaign"].where(
    df["lead_utm_campaign"].isin(top_values),
    "other"
)

In [55]:
df["lead_utm_campaign_grouped"].value_counts()

lead_utm_campaign_grouped
other        13772
116543546      685
115509254      560
118448375      376
700615408      339
115367108      272
702829935      215
118075886      197
118108441      170
702103466      169
119815667      153
114317039      139
118751984      138
114252495      133
702279529      133
119707922      130
701745533      110
702467942      100
702013262       96
702239869       96
114824436       93
118740070       91
702233419       90
702467952       89
705549532       87
702176041       86
705835502       78
119948969       75
702417626       75
705007398       74
702103459       66
Name: count, dtype: int64

In [56]:
df = df.drop(columns=["lead_utm_campaign"])

*********

In [57]:


df[~df["contact_Источник трафика"].isna()]["contact_Источник трафика"].value_counts()

contact_Источник трафика
dzen.ru                          1047
Yandex Direct                     799
m.dzen.ru                         350
Yandex                            145
artraid.promo.page                144
                                 ... 
punkt-a.info                        1
allcafe.ru                          1
93.ru                               1
uznayvse.ru                         1
Переходы по ссылкам на сайтах       1
Name: count, Length: 469, dtype: int64

In [58]:
df["traffic_source_missing"] = df["contact_Источник трафика"].isna().astype(int)

top_k = 30

top_values = (
    df["contact_Источник трафика"]
    .value_counts()
    .head(top_k)
    .index
)

df["traffic_source_grouped"] = df["contact_Источник трафика"].where(
    df["contact_Источник трафика"].isin(top_values),
    "other"
)

In [59]:
df["traffic_source_grouped"].value_counts()

traffic_source_grouped
other                  15531
dzen.ru                 1047
Yandex Direct            799
m.dzen.ru                350
Yandex                   145
artraid.promo.page       144
Прямой заход              91
Google                    90
ria.ru                    67
gismeteo.ru               60
m.ura.news                47
лечу-варикоз.рф           45
rlsnet.ru                 44
pogoda7.ru                40
mail.ru                   35
tv.yandex.ru              31
pravda.ru                 28
vidal.ru                  28
progoroduhta.ru           28
Переходы по рекламе       25
otzovik.com               25
mail.yandex.ru            23
kp.ru                     21
life.ru                   20
hibiny.ru                 20
ok.ru                     18
e.mail.ru                 18
news.ru                   18
samaragovorit.ru          18
progorod43.ru             16
news-dagestan.ru          15
Name: count, dtype: int64

In [60]:
df = df.drop(columns=["contact_Источник трафика"])

*************

In [61]:
# Иными словами, это старт этапа внутренней обработки заказа, до доставки
# как только заказ дошёл до доставки, ты уже знаешь, что он прошёл половину воронки — и это делает
# задачу проще для модели нечестным образом

#переход в сборку — тоже событие после оформления в день оформления
# неизвестно, будет ли сборка и когда она произойдёт

"""df["lead_created_dt"] = pd.to_datetime(df["lead_created_at"], unit="s", utc=True)
df["lead_Дата перехода в Сборку"] = pd.to_datetime(df["lead_Дата перехода в Сборку"], unit="s", utc=True)
start = pd.Timestamp("2025-03-01")
end   = pd.Timestamp("2026-03-29")

date_cols = [
    "lead_created_dt",
    "lead_Дата перехода в Сборку"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

    df.loc[
        (df[col] < start) | (df[col] > end),
        col
    ] = pd.NaT

df["lead_to_assembly_days"] = (
    df["lead_Дата перехода в Сборку"] - df["lead_created_dt"]
).dt.total_seconds() / 86400"""

'df["lead_created_dt"] = pd.to_datetime(df["lead_created_at"], unit="s", utc=True)\ndf["lead_Дата перехода в Сборку"] = pd.to_datetime(df["lead_Дата перехода в Сборку"], unit="s", utc=True)\nstart = pd.Timestamp("2025-03-01")\nend   = pd.Timestamp("2026-03-29")\n\ndate_cols = [\n    "lead_created_dt",\n    "lead_Дата перехода в Сборку"\n]\n\nfor col in date_cols:\n    df[col] = pd.to_datetime(df[col], errors="coerce")\n\n    df.loc[\n        (df[col] < start) | (df[col] > end),\n        col\n    ] = pd.NaT\n\ndf["lead_to_assembly_days"] = (\n    df["lead_Дата перехода в Сборку"] - df["lead_created_dt"]\n).dt.total_seconds() / 86400'

In [62]:
# --- 2. delay (сборка - создание) ---
# !!!!!!!!!!!!!!!!!!!!!!!
# lead_created → (прогноз здесь) → сборка → доставка → получение → оплата
# Предсказываем решение клиента, но даём модели информацию о том, как далеко заказ
# уже продвинулся — это и есть leakage.
"""
df["lead_created_dt"] = pd.to_datetime(df["lead_created_at"], unit="s", utc=True)
df["lead_Дата перехода в Сборку"] = pd.to_datetime(df["lead_Дата перехода в Сборку"], unit="s", utc=True)

df["lead_to_assembly_days"] = (
    df["lead_Дата перехода в Сборку"] - df["lead_created_dt"]
).dt.total_seconds() / 86400

# --- 3. исправление timezone-сдвига (+1 день) ---
df["lead_to_assembly_days"] = df["lead_to_assembly_days"] + 1

# --- 4. флаг события ---
df["assembly_flag"] = df["lead_Дата перехода в Сборку"].notna().astype(int)

# --- 5. контроль хвоста ---
df["lead_to_assembly_long"] = (df["lead_to_assembly_days"] > 30).astype(int)

df["lead_to_assembly_days"] = df["lead_to_assembly_days"].clip(upper=30)

# --- 6. обработка пропусков ---
df["lead_to_assembly_days"] = df["lead_to_assembly_days"].fillna(-1)

"""

# --- 7. удаление исходного признака ---
df = df.drop(columns=["lead_Дата перехода в Сборку"])

*************************

In [63]:
df[~df["lead_utm_term"].isna()]["lead_utm_term"].value_counts()



lead_utm_term
dem_otek_clip1_otek                       1665
dem_varicose_clip1_varikoz                1057
yur_sustavy_applicator_landing-artraid     855
dem_sustav_clip002                         799
yur_sleep_maska_sleep                      729
                                          ... 
dem2_varicose_Confeti_varikoz                1
999999                                       1
sustavy_clip                                 1
catalog                                      1
nim_varicose_bandage_varikoz_clip_1          1
Name: count, Length: 162, dtype: int64

In [64]:
df["lead_utm_term"] = df["lead_utm_term"].replace(
    ["Неизвестно"],
    np.nan
)

df["utm_term_missing"] = df["lead_utm_term"].isna().astype(int)

# --- 3. top-K + other ---
top_k = 40
top_values = df["lead_utm_term"].value_counts().head(top_k).index

df["utm_term_grouped"] = df["lead_utm_term"].where(
    df["lead_utm_term"].isin(top_values),
    "other"
)
# Можно было поиграться с frequency encoding, но это уже тюнинг

In [65]:
df = df.drop(columns=["lead_utm_term"])

In [66]:
df["utm_term_grouped"].value_counts()

utm_term_grouped
other                                              7106
dem_otek_clip1_otek                                1665
dem_varicose_clip1_varikoz                         1057
yur_sustavy_applicator_landing-artraid              855
dem_sustav_clip002                                  799
yur_sleep_maska_sleep                               729
yur_varicose_bandage_landing-artraid-2              714
dem_sleep_clip                                      650
yur_sustavy_applicator_poyasnica                    521
but_varicose_bandage_varikoz_clip                   393
nim_varicose_bandage_varikoz_clip                   368
nim_varicose_bandage_landing-artraid-2              347
yur_varicose_bandage_varikoz_clip                   345
nim_sleep_maska_sleep                               315
dem_davlenie2                                       190
dem_sustav_clip                                     185
dem_ssz                                             178
but_varicose_bandage_landing-ar

*********************

In [67]:
len(df[~df["lead_utm_sky"].isna()]["lead_utm_sky"])

11492

In [68]:
df[~df["lead_utm_sky"].isna()]["lead_utm_sky"].value_counts().head(30)


lead_utm_sky
---autotargeting                             9467
{keyword}                                      90
p=but                                          80
artraid                                        45
длительное нарушение сна                       29
нарушение сна человека                         25
артрейд                                        14
варикоз                                        14
нарушение сна раздражительность                13
artraid микросферы                             12
лечение варикоза                               10
флеболог и узи вен                             10
варикоз лазерная                                9
artraid официальный сайт                        9
undefined                                       9
флеболог лечит варикоз                          9
прием флеболога                                 9
микросферы артрейд                              8
варикоз на ногах у мужчин                       8
артраид микросферы купить            

In [69]:
df["lead_utm_sky"] = df["lead_utm_sky"].replace(
    ["{keyword}"],
    np.nan
)

# Базовый missing-флаг
df["utm_sky_missing"] = df["lead_utm_sky"].isna().astype(int)

# отдельный тип трафика
df["utm_sky_autotarget"] = (df["lead_utm_sky"] == "---autotargeting").astype(int)

# выделим бренд
df["utm_sky_brand"] = df["lead_utm_sky"].str.contains(
    "artraid|артрейд",
    case=False,
    na=False
).astype(int)

# семантика запроса
df["utm_sky_varicose"] = df["lead_utm_sky"].str.contains(
    "варикоз|вены|флеболог",
    case=False,
    na=False
).astype(int)

df["utm_sky_sleep"] = df["lead_utm_sky"].str.contains(
    "сон|sleep",
    case=False,
    na=False
).astype(int)

In [70]:
df["utm_sky_autotarget"].value_counts()

utm_sky_autotarget
1    9467
0    9420
Name: count, dtype: int64

In [71]:
df = df.drop(columns=["lead_utm_sky"])

************************

In [72]:
# rejected_dt - утечка, просто удаляем
df = df.drop(columns=["rejected_ts"])

******************

In [73]:
df[~df["lead_group"].isna()]["lead_group"].value_counts()


lead_group
yur     4674
but     1211
gus       19
inr        7
imp        3
kru        2
nil        2
nim        2
dem2       2
mea        1
dem        1
Name: count, dtype: int64

In [74]:
# флаг пропуска
df["lead_group_missing"] = df["lead_group"].isna().astype(int)

# топ-2 группы + other
main_groups = ["yur", "but"]

df["lead_group_grouped"] = df["lead_group"].where(
    df["lead_group"].isin(main_groups),
    "other"
)

# удаление исходного столбца


In [75]:
df["lead_group_grouped"].value_counts()

lead_group_grouped
other    13002
yur       4674
but       1211
Name: count, dtype: int64

In [76]:
df = df.drop(columns=["lead_group"])

***************

In [77]:
len(df[~df["lead_Проблема"].isna()]["lead_Проблема"])



18860

In [78]:
# missing-флаг
df["problem_missing"] = df["lead_Проблема"].isna().astype(int)

# --- 3. укрупнение категорий ---
main_groups = [
    "Суставы и позвоночник",
    "Варикоз",
    "Сердечно-сосудистые заболевания",
    "Бессоница",
    "Головные боли",
    "Отеки",
    "Зрительная система",
    "Давление",
    "Инсульт",
    "Боли и тяжесть в ногах"
]

df["problem_grouped"] = df["lead_Проблема"].where(
    df["lead_Проблема"].isin(main_groups),
    "other"
)



In [79]:
df["problem_grouped"].value_counts()

problem_grouped
Суставы и позвоночник              8347
Варикоз                            5403
Сердечно-сосудистые заболевания    1238
Бессоница                          1180
Головные боли                       643
other                               460
Отеки                               459
Зрительная система                  382
Давление                            303
Инсульт                             240
Боли и тяжесть в ногах              232
Name: count, dtype: int64

In [80]:
df = df.drop(columns=["lead_Проблема"])

****************

In [81]:
df[~df["lead_Высота"].isna()]["lead_Высота"].count()

np.int64(12100)

In [82]:
col = "lead_Высота" # более жирный признак его оставляем

# 1. Флаг пропуска
df["lead_height_known"] = df[col].notna().astype(int)

# 2. Клип
q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1
upper = q3 + 2 * iqr

df["lead_height_clean"] = df[col].clip(upper=upper)

# 3. Лог
df["lead_height_log"] = np.log1p(df["lead_height_clean"])
df["lead_height_log"] = df["lead_height_log"].fillna(0)

# 4. Бин
def height_bin(x):
    if pd.isna(x):
        return "unknown"
    elif x <= 4:
        return "small"
    elif x <= 12:
        return "medium"
    else:
        return "large"

df["lead_height_bin"] = df["lead_height_clean"].apply(height_bin)

# 5. Удаляем лишнее
df = df.drop(columns=[
    col,
    "lead_height_clean"
])


In [83]:
df_model = df[df["buyout_flag"].notna()].copy()

df_model["buyout_flag"] = df_model["buyout_flag"].astype(int)
df_model["buyout_flag"].unique()

array([1, 0])

In [84]:
roc_auc_score(
    df_model["buyout_flag"],
    df_model["lead_height_log"].astype(float)
)

0.5357214213987039

*********************

In [85]:
df[~df["lead_Стоимость доставки"].isna()]["lead_Стоимость доставки"].count()

np.int64(3062)

In [86]:
col = "lead_Стоимость доставки"
df[col]

0        NaN
1        NaN
2        NaN
3        NaN
4        NaN
        ... 
18882    NaN
18883    NaN
18884    NaN
18885    NaN
18886    NaN
Name: lead_Стоимость доставки, Length: 18887, dtype: str

In [87]:

df[col] = (
    df[col]
    .astype(str)
    .str.strip()
    .replace({"-": None, "": None})          # убираем мусор
    .str.replace(",", ".", regex=False)      # 515,06 → 515.06
)

df[col] = pd.to_numeric(df[col], errors="coerce")

df[col].describe()

count     3061.000000
mean      1096.991431
std        595.882051
min        346.000000
25%        777.860000
50%        966.350000
75%       1276.020000
max      16123.150000
Name: lead_Стоимость доставки, dtype: float64

In [88]:
df["delivery_cost_missing"] = df[col].isna().astype(int)
df["delivery_cost_log"] = np.log1p(df[col])
df["delivery_cost_log"] = df["delivery_cost_log"].fillna(-1)
df = df.drop(columns=[col])

*******************

In [89]:

# =========================
# 1. ПОДГОТОВКА
# =========================
work = df[["lead_id", "contact_id", "sale_ts", "lead_price"]].copy()

# datetime
work["sale_dt"] = pd.to_datetime(work["sale_ts"], unit="s", errors="coerce")

# сумма заказа
work["order_amount"] = pd.to_numeric(work["lead_price"], errors="coerce").fillna(0)
work.loc[work["order_amount"] < 0, "order_amount"] = 0

# убираем мусор
work = work[work["contact_id"].notna() & work["sale_dt"].notna()].copy()

# сортировка
work = work.sort_values(["contact_id", "sale_dt", "lead_id"])

# =========================
# 2. LTV + COUNT
# =========================
# накопленная сумма
cum = work.groupby("contact_id")["order_amount"].cumsum()

# LTV до текущего заказа
work["LTV_as_of_sale"] = (
    work.groupby("contact_id")[cum.name]
    .shift(1)
    .fillna(0)
)

# количество прошлых заказов
work["past_orders_cnt"] = work.groupby("contact_id").cumcount()

# бинарный признак
work["is_repeat_client_as_of_sale"] = (work["past_orders_cnt"] > 0).astype(int)

# =========================
# 3. ОСТАВЛЯЕМ ТОЛЬКО НУЖНОЕ
# =========================
features = work[
    [
        "lead_id",
        "LTV_as_of_sale",
        "past_orders_cnt",
        "is_repeat_client_as_of_sale",
    ]
]
#LTV_as_of_sale → сколько клиент уже потратил
#past_orders_cnt → сколько заказов было
#is_repeat_client_as_of_sale → новый / повторный
# =========================
# 4. MERGE ОБРАТНО
# =========================
df = df.merge(features, on="lead_id", how="left")

# fill для строк без истории
df["LTV_as_of_sale"] = df["LTV_as_of_sale"].fillna(0)
df["past_orders_cnt"] = df["past_orders_cnt"].fillna(0)
df["is_repeat_client_as_of_sale"] = df["is_repeat_client_as_of_sale"].fillna(0)

# =========================
# 5. SANITY CHECK
# =========================
assert (df["LTV_as_of_sale"] >= 0).all()

***************************

In [90]:
# ======================================================
# 4. FEATURE ENGINEERING
#
# ======================================================
# shipping cost
if "lead_shipping_cost" in df.columns:
    df["lead_shipping_cost"] = df["lead_shipping_cost"].replace(-1, 0)
    df["lead_shipping_cost_log"] = np.log1p(df["lead_shipping_cost"])

# price
if "lead_price" in df.columns:
    df["lead_price_log"] = np.log1p(df["lead_price"])
    df["expensive_order"] = (df["lead_price"] > df["lead_price"].median()).astype(int)
    df["price_bin"] = pd.qcut(df["lead_price"], 5, duplicates="drop")

# price / delivery ratio
if {"lead_price", "lead_shipping_cost"}.issubset(df.columns):
    df["price_delivery_ratio"] = df["lead_price"] / (df["lead_shipping_cost"] + 1)
    df["pdr_bin"] = pd.qcut(df["price_delivery_ratio"], 5, duplicates="drop")

    df["pdr_optimal"] = (
        (df["price_delivery_ratio"] > 3230) &
        (df["price_delivery_ratio"] <= 7290)
    ).astype(int)

# pillow interaction
if {"expensive_order", "lead_has_pillow"}.issubset(df.columns):
    df["expensive_pillow"] = df["expensive_order"] * df["lead_has_pillow"]

# night / weekend
if "lead_creation_date_sin" in df.columns:
    df["night_order"] = (df["lead_creation_date_sin"] < -0.5).astype(int)

if "lead_created_hour" in df.columns:
    df["is_night"] = df["lead_created_hour"].between(0, 6).astype(int)

if "lead_creation_date_dayofweek" in df.columns:
    df["is_weekend_creation"] = df["lead_creation_date_dayofweek"].isin([5, 6]).astype(int)

# delivery cost high
if "lead_shipping_cost_log" in df.columns:
    df["delivery_cost_high"] = (
        df["lead_shipping_cost_log"] > df["lead_shipping_cost_log"].median()
    ).astype(int)
    df["delivery_bin"] = pd.qcut(df["lead_shipping_cost"], 5, duplicates="drop")

# discount
if {"lead_discount", "lead_price"}.issubset(df.columns):
    df["has_discount"] = (df["lead_discount"] > 0).astype(int)
    df["discount_ratio"] = df["lead_discount"] / (df["lead_price"] + 1)
    df["high_discount"] = (df["discount_ratio"] > df["discount_ratio"].median()).astype(int)

if {"expensive_order", "high_discount"}.issubset(df.columns):
    df["expensive_high_discount"] = df["expensive_order"] * df["high_discount"]

# paid traffic interaction
if {"expensive_order", "is_paid_traffic"}.issubset(df.columns):
    df["expensive_paid"] = df["expensive_order"] * df["is_paid_traffic"]

# items count interaction
if {"expensive_order", "lead_items_count"}.issubset(df.columns):
    df["expensive_multi"] = df["expensive_order"] * (df["lead_items_count"] > 1).astype(int)

# main pipeline
if "lead_pipeline_id" in df.columns:
    top_pipeline = 6892026
    df["is_main_pipeline"] = (df["lead_pipeline_id"] == top_pipeline).astype(int)

In [91]:
#df["lead_created_at"] = df["lead_created_at"].fillna(0)
#df["handed_to_delivery_ts"] = df["handed_to_delivery_ts"].fillna(0)

df = df.drop(columns=["lead_created_at", "lead_id", "contact_id", "outcome_unknown", "sale_ts", "lead_price"])



In [92]:
df["row_id"]

0            0
1            1
2            2
3            3
4            4
         ...  
18882    18882
18883    18883
18884    18884
18885    18885
18886    18886
Name: row_id, Length: 18887, dtype: int64

In [93]:
df.to_csv(OUTPUT_DIR / "X_block_5.csv", index=False)
print("Saved:.")
print(OUTPUT_DIR / "X_block_5.csv")

Saved:.
..\..\notebook_outputs\group_5\X_block_5.csv
